In [0]:
%run ./00_config

In [0]:
#setup and log function
import json
from pyspark.sql import functions as F

def log_run(status, detail):
    (spark.createDataFrame([("", status, json.dumps(detail))],
                           "file_name string, status string, detail string")
       .select(F.current_timestamp().alias("run_at"), F.lit("silver_to_gold").alias("stage"),
               "file_name", "status", "detail")
       .write.mode("append").saveAsTable("nhsae.ops.run_log"))

In [0]:
#The gold national performance table
spark.sql("""
CREATE OR REPLACE TABLE nhsae.gold.national_by_type
COMMENT 'National A&E 4-hour performance. Grain: month x department type x route'
AS
SELECT
  period_date, dept_type, route,
  SUM(attendances) AS attendances,
  SUM(over_4hr)    AS over_4hr,
  ROUND(100 * (1 - SUM(over_4hr) / NULLIF(SUM(attendances), 0)), 2) AS pct_within_4hr
FROM nhsae.silver.ae_by_type
GROUP BY period_date, dept_type, route
""")
display(spark.table("nhsae.gold.national_by_type").orderBy("period_date", "route", "dept_type"))

In [0]:
#Gold table; performance across trust
spark.sql("""
CREATE OR REPLACE TABLE nhsae.gold.trust_monthly
COMMENT 'Provider-level 4-hour and 12-hour DTA performance. Grain: org x month'
AS
WITH att AS (
  SELECT period_date, org_code,
         SUM(attendances) AS attendances,
         SUM(over_4hr)    AS over_4hr
  FROM nhsae.silver.ae_by_type
  GROUP BY period_date, org_code
)
SELECT
  o.period_date, o.org_code, o.org_name, o.parent_org,
  CASE WHEN o.org_code RLIKE '^R[A-Z0-9]{2}$' THEN 'NHS trust' ELSE 'Other provider' END AS org_type,
  a.attendances, a.over_4hr,
  ROUND(100 * (1 - a.over_4hr / NULLIF(a.attendances, 0)), 2) AS pct_within_4hr,
  o.dta_12hr_plus, o.emergency_admissions_ae,
  ROUND(1000 * o.dta_12hr_plus / NULLIF(o.emergency_admissions_ae, 0), 1) AS dta_12hr_per_1000_adm
FROM nhsae.silver.ae_org_monthly o
JOIN att a USING (period_date, org_code)
""")
display(spark.table("nhsae.gold.trust_monthly").orderBy(F.desc("attendances")))

In [0]:
#sanity checks and headline
n_national = spark.table("nhsae.gold.national_by_type").count()
n_trust    = spark.table("nhsae.gold.trust_monthly").count()
periods    = [str(r[0]) for r in spark.sql(
    "SELECT DISTINCT period_date FROM nhsae.gold.trust_monthly ORDER BY 1").collect()]

headline = spark.sql("""
  SELECT period_date,
         SUM(attendances) AS attendances,
         ROUND(100 * (1 - SUM(over_4hr) / SUM(attendances)), 1) AS pct_within_4hr
  FROM nhsae.gold.national_by_type GROUP BY period_date ORDER BY period_date
""")
display(headline)



In [0]:
#log and exit
result = {"status": "gold", "periods": periods,
          "national_rows": n_national, "trust_rows": n_trust}
log_run("gold", result)
dbutils.notebook.exit(json.dumps(result))